In [ ]:
import numpy as np
import pandas as pd
import polars as pl
import matplotlib.pyplot as plt
import seaborn as sns
import optuna
import math
from sklearn.calibration import calibration_curve
from sklearn.metrics import (
    confusion_matrix,
    f1_score,
    roc_auc_score,
    roc_curve,
)
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.preprocessing import StandardScaler
 
from Extraction import extract
from Transformation_Pretraitement import preprocessing_polars
from inceptionTimeModified import (
    evaluate_on_test,
    load_model_from_checkpoint,
    predict_proba,
    train_inception_time,
)
import utils_inception as ui

In [ ]:
pl.Config.set_tbl_cols(-1)

 ##### Dataframe statique

In [ ]:
path = "../Datasets/clean_full_static_ano.parquet"
df_static = pl.read_parquet(path)
df_static = df_static.with_columns(pl.col("encounterId").cast(pl.Int32))

In [ ]:
print("nombre d'outliers ",len(df_static.filter(pl.col("deces_datediff_days") < -1)))

In [ ]:
outliers = df_static.filter(pl.col("deces_datediff_days") < -1)

In [ ]:
n_total = len(df_static)
n_removed = len(df_static.filter(pl.col("deces_datediff_days") < -1))
n_mapped_to_zero = len(
    df_static.filter(pl.col("deces_datediff_days").is_between(-1, 0))
)
n_null = len(df_static.filter(pl.col("deces_datediff_days").is_null()))

print(f"Total: {n_total}")
print(f"Supprimées (< -1): {n_removed}")
print(f"Remappées à 0 (entre -1 et 0): {n_mapped_to_zero}")
print(f"Null conservés: {n_null}")
print(f"% supprimé: {100 * n_removed / n_total:.4f}%")



df_filtered = (
    df_static.with_columns(
        pl.when(pl.col("deces_datediff_days").is_between(-1, 0))
        .then(0)
        .otherwise(pl.col("deces_datediff_days"))
        .alias("deces_datediff_days")
    )
    .filter(
        (pl.col("deces_datediff_days") >= 0) |
        (pl.col("deces_datediff_days").is_null())
    )
)

print("Avant :", len(df_static))
print("Après :", len(df_filtered))
print("Différence :", len(df_static) - len(df_filtered))

In [ ]:
summary = df_static.select(
    [
        pl.len().alias("n_total"),
        (pl.col("deces_datediff_days") < -1).sum().alias("n_removed"),
        pl.col("deces_datediff_days").is_between(-1, 0).sum().alias("n_set_to_zero"),
        pl.col("deces_datediff_days").is_null().sum().alias("n_null"),
    ]
)

print(summary)

In [ ]:
# df_static = df_static.with_columns(
#     pl.when(pl.col("deces_datediff_days").is_between(-1, 0))
#     .then(0)
#     .otherwise(pl.col("deces_datediff_days"))
#     .alias("deces_datediff_days")
# ).filter(
#     (pl.col("deces_datediff_days") >= 0) |
#     (pl.col("deces_datediff_days").is_null())
# )


In [ ]:
df_static = df_static.with_columns(
    pl.when(pl.col("deces_datediff_days").is_between(-1, 0))
    .then(0)
    .otherwise(pl.col("deces_datediff_days"))
    .alias("deces_datediff_days")
)


In [ ]:
df_static["isDeceased"].describe()

In [ ]:
import matplotlib.pyplot as plt
plt.hist(df_static["deces_datediff_days"], bins=1000)
plt.title("Distribution de décès")
plt.show()

##### Dataframe dynamique

In [ ]:
path = "../Datasets/df_with_calculated_features.parquet"
df_test = extract.extract_data_survie(path)

#### Transformation_Prétraitement

Il faudra changer hour_offset pour pouvoir prendre une date fixe et non juste un temps en arrière

ajout de la colonne age qui est dans le thesaurus

In [ ]:
df_test = df_test.join(
    df_static[["encounterId", "age"]],
    on="encounterId",
    how="left"
)

On utilise ici le preprocessing de Gabrielle mais avec l'optimisation polars réalisée par mes soins, puisque l'ancien code mettait beaucoup trop de temps à tourner.

In [ ]:
df_clean = preprocessing_polars.prepare_data(df_test,0)

In [ ]:
df_clean

In [ ]:
df_with_idx = df_clean.with_row_index("idx")
 
idx = (

    df_with_idx

    .filter(pl.col("fio2_corr").is_null())

    .select("idx")

)
 
print(len(idx))

df_clean.filter(pl.col("fio2_corr").is_null())

Il faut rajouter isDeceased sinon on n'a pas de Y

In [ ]:
outliers_complete = outliers.join(df_clean, on = "encounterId", how = "inner")
test = outliers_complete.select("encounterId", "heure_calibree", "deces_datediff_days")
for row in test.iter_rows():
    print(row)

In [ ]:
test = outliers_complete.select("encounterId", "heure_calibree", "deces_datediff_days")
eId = [row["encounterId"] for row in test.iter_rows(named = True)]
eId = list(set(eId))
print(eId)
for ide in eId:
    test22 = outliers_complete.filter(pl.col("encounterId") == ide)
    x = test22.columns
    x.remove("heure_calibree")
    x.remove("transition_units")
    x.remove("icu_DA")
    x.remove("icu_actes")
    x.remove("icu_ghm")
    x.remove("encounterId")
    xx = [row["heure_calibree"] for row in test22.iter_rows(named = True)]
    for col in x:
        yy = [row[col] for row in test22.iter_rows(named = True)] 
        print(xx)
        print(col, yy)
        plt.plot(xx, yy)
        plt.xlabel("heure_calibree")
        plt.ylabel(x)
        plt.title(f"{col} pour l'id {ide} en fonction de l'heure calibrée")
        plt.savefig(f"outputs/{ide}_{col}_HC")
        plt.show()

In [ ]:
df_clean

In [ ]:
print(len(df_clean.filter(pl.col("heure_calibree") > 0)))

In [ ]:
test = df_clean.filter(pl.col("heure_calibree") > 0)

In [ ]:
encounterIdPb = [row["encounterId"] for row in test.iter

In [ ]:
    df_clean2 = df_clean.with_columns(
    (
        - pl.col("heure_calibree").min().over("encounterId")
    ).alias("duree_sejour")
    )

In [ ]:
df_clean2["duree_sejour"].describe()

In [ ]:
duree_sejour = df_clean2["duree_sejour"].to_list()
duree_sejour.remove(max(duree_sejour))

In [ ]:
ma_liste = np.array(duree_sejour)
import matplotlib.pyplot as plt

x = ma_liste
q99 = np.quantile(x, 0.99)

x_filtered = x[x <= q99]

plt.hist(x_filtered, bins=200)
plt.show()

In [ ]:
from matplotlib.ticker import MaxNLocator
plt.figure(0,(60,10))
plt.hist(x_filtered, bins=200)

plt.gca().xaxis.set_major_locator(MaxNLocator(nbins=200))
plt.show()


from matplotlib.ticker import MultipleLocator
plt.figure(1,(180,10))
plt.hist(x_filtered, bins=400)

plt.gca().xaxis.set_major_locator(MultipleLocator(5))  # tick tous les 20
plt.savefig("test")
plt.show()